# Few-Shot Prompting for Sentiment Classification

**Course:** Natural Language Processing · Unit 4 — Prompt Engineering  
**Institution:** Universidad Tecnológica de Bolívar  
**Reference:** Jurafsky, D. & Martin, J. H. (2025). *Speech and Language Processing* (3rd ed.), Ch. 12 — Prompting and In-Context Learning  
**Python compatibility:** 3.10 · 3.11 · 3.12 · 3.13

---

## Learning Objectives

1. Understand how in-context exemplars guide an LLM's output format
2. Build a few-shot sentiment classifier using the OpenAI Completions API
3. Implement a pattern-matching simulation to demonstrate the CoT concept without an API key
4. Discuss the trade-offs between few-shot size and latency/cost


---

## 1. Environment Setup

Install once in your terminal:

```bash
pip install openai
```


In [ ]:
# pip install openai  # run once in terminal
import os

import openai

openai.api_key = os.environ.get("OPENAI_API_KEY", "your-openai-api-key-here")


---

## 2. Few-Shot Sentiment Classification via API

The prompt contains three labelled examples (positive / negative / neutral) followed  
by the target text. The model extracts the pattern and applies it without fine-tuning.


In [ ]:
def classify_sentiment(text: str) -> str:
    """Classify the sentiment of *text* using few-shot prompting."""
    prompt = (
        "Classify the sentiment of the following phrases as positive, negative, or neutral.\n\n"
        "Phrase: I love this new phone.\n"
        "Sentiment: positive\n\n"
        "Phrase: This product is terrible and does not work.\n"
        "Sentiment: negative\n\n"
        "Phrase: The sky today is cloudy.\n"
        "Sentiment: neutral\n\n"
        f"Phrase: {text}\n"
        "Sentiment:"
    )

    response = openai.Completion.create(
        model="gpt-3.5-turbo-instruct",
        prompt=prompt,
        max_tokens=10,
        n=1,
        stop=["\n"],
    )
    return response.choices[0].text.strip()


samples = [
    "I am very happy with my purchase.",
    "I feel very disappointed.",
    "The meeting has been rescheduled.",
]
for sample in samples:
    print(f"Text: {sample}")
    print(f"Sentiment: {classify_sentiment(sample)}")
    print()


> **Output interpretation:** Each call sends the three exemplars plus the new text to the model; the model returns one of 'positive', 'negative', or 'neutral'. Adding more diverse exemplars (e.g. sarcasm, irony) improves robustness but increases prompt length and cost per request.


---

## 3. Offline Simulation (No API Required)

This section demonstrates the *concept* of few-shot prompting using a pure-Python  
dictionary lookup — no API key needed. The model is replaced by an exact-match  
lookup over the training exemplars.


In [ ]:
def simulate_few_shot_classification(prompt_examples: str) -> None:
    """Print the exemplars and classify animal types by exact lookup.

    Args:
        prompt_examples: Newline-separated 'Animal: Type' pairs.
    """
    examples: dict[str, str] = {}

    print("Provided exemplars:")
    for line in prompt_examples.strip().split("\n"):
        if ":" in line:
            animal, kind = line.split(":", 1)
            examples[animal.strip().lower()] = kind.strip().lower()
            print(f"  - {animal.strip()}: {kind.strip()}")
    print("-" * 40)

    test_inputs = ["dog", "parrot", "dolphin"]
    for animal in test_inputs:
        if animal.lower() in examples:
            print(f"'{animal}' → '{examples[animal.lower()]}' (matched from exemplars)")
        else:
            print(f"'{animal}' → 'unknown' (not in exemplars — a real LLM would generalise)")


exemplar_prompt = (
    "Dog: Mammal\n"
    "Canary: Bird\n"
    "Snake: Reptile\n"
    "Cat: Mammal\n"
    "Fish: Aquatic"
)
simulate_few_shot_classification(exemplar_prompt)


> **Output interpretation:** The simulation shows that the 'parrot' and 'dolphin' inputs are not matched (they were not in the exemplars), returning 'unknown'. A real LLM would generalise from the patterns in the exemplars — e.g. recognising 'parrot' as a 'Bird' because it shares features with 'Canary'. This illustrates the power of few-shot prompting over simple lookup tables.


---

## Summary

| Aspect | Few-shot prompting | Fine-tuning |
|---|---|---|
| Training data needed | 3–20 examples in the prompt | Hundreds–thousands of labelled samples |
| Latency | Longer prompt, slower inference | Faster inference (shorter prompt) |
| Cost | Higher token cost per call | High upfront training cost |
| Flexibility | Change examples at runtime | Requires re-training to update |
